# Try to resolve the problem of stress tanckling 

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))
from config import paths

In [3]:
traj = paths.get_trajectory_path
figto = paths.get_figure_path
data = paths.get_data_path

In [4]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['savefig.dpi'] = 300
matplotlib.rcParams['font.size'] = 12
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
from collections import defaultdict

from tools.custom_hbond_analysis import HydrogenBondAnalysis as HBA
from tools.zeta_order_parameter import ZetaOrderParameter as ZOP

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
u = mda.Universe(traj("tip4p-ice-225K.data"), traj("peratom_stress_5e-5_225.lammpstrj"),
                 format="LAMMPSDUMP",
                 additional_columns=["c_peratom[4]"])
u.trajectory[0]
pxy = u.trajectory.ts.data["c_peratom[4]"]  # shape (n_atoms,) – it just works

/usr/local/lib/python3.12/dist-packages/MDAnalysis/coordinates/LAMMPS.py:749: UserWarning: Reader has no dt information, set to 1.0 ps
  ts.data["time"] = step_num * ts.dt


In [6]:
print("pxy shape:", pxy.shape)

pxy shape: (12288,)


## Another solution

In [ ]:
from tools.stress_lammps import StressDumpReader, MesoscopicAnalysis
u2 = mda.Universe(traj("tip4p-ice-225K.data"), traj("peratom_stress_5e-5_225.lammpstrj"),
                    format=StressDumpReader, dt=0.025, stress_prefix='c_peratom', lammps_units='real')
u2.trajectory[0]
pxy_frame0 = u2.trajectory.ts.data['stress_xy'] # shape (n_atoms,) – it just works
stress_all = u2.trajectory.ts.data['stress'] # shape (n_atoms, 6) – it just works
print("pxy shape:", pxy.shape)
print("stress shape:", stress_all.shape)

In [11]:
from tools.stress_lammps import stress_universe
u = stress_universe(traj("tip4p-ice-225K.data"), traj("peratom_stress_5e-5_225.lammpstrj"),
                    stress_prefix='c_peratom', dt=0.025)

/usr/local/lib/python3.12/dist-packages/MDAnalysis/coordinates/LAMMPS.py:851: UserWarning: Some of the additional columns are not present in the file, they will be ignored
  warnings.warn(


In [12]:
ma = MesoscopicAnalysis(
    universe=u,
    temperature=225,
    oxygen_sel='type 1',
    stress_component='xy',
    tau_max=2000,
)

ma.run()

eta = ma.results.viscosity_GK
D = ma.results.diffusion_msd

/usr/local/lib/python3.12/dist-packages/MDAnalysis/coordinates/LAMMPS.py:851: UserWarning: Some of the additional columns are not present in the file, they will be ignored
  warnings.warn(



MESOSCOPIC ANALYSIS RESULTS
  Temperature            : 225.0 K
  Stress component       : P_xy
  Frames analysed        : 801
  τ_max for ACF          : 2000 frames (50.0 ps)

  Green-Kubo viscosity   : 242.8149 mPa·s
  Diffusion (MSD)        : 1.4906e-08 m²/s
  ⟨P_xy⟩ (mean stress)   : 205333744.3668 Pa
  σ(P_xy) (fluctuation)  : 49837747.3382 Pa



/root/SupercooledWater/tools/stress_lammps.py:586: UserWarning: Velocities not found in trajectory.  VACF-based diffusion will be NaN.  Include vx/vy/vz columns in the LAMMPS dump to enable this.
  warnings.warn(
